In [ ]:
# class MockUploadFile:
#     def __init__(self, filename, content: bytes):
#         self.filename = filename
#         self._content = content
#     async def read(self):
#         return self._content

# with open("E:\AI Projects\doc_files\CE1_EN.docx", "rb") as f:
#     file_bytes = f.read()

# file = MockUploadFile(r"E:\AI Projects\doc_files\CE1_EN.docx", file_bytes)
# docs, loaders = await load_documents_from_files_or_zip(uploaded_files = [file], 
#                                  file_path = [r"E:\AI Projects\doc_files\CE1_EN.docx"],
#                                  FileNames = [r"E:\AI Projects\doc_files\CE1_EN.docx"],
#                                  file_hashes = ["123abc"])
# print(docs)
# print(loaders)

In [ ]:
from req import DOCXLoader

with open(r"E:\AI Projects\doc_files\CE1_EN.docx", "rb") as f:
    doc_bytes = f.read()

loader = DOCXLoader(file_content=doc_bytes)
paragraphs = loader.load()
list_completions = [a.page_content for a in paragraphs]
print(list_completions)

In [ ]:
import requests
from qdrant_client import QdrantClient
from openai import OpenAI
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

client = QdrantClient(url="http://localhost:6333")
client_opeai = OpenAI(
        base_url = "http://172.30.14.32:5000/v1",
        api_key = os.getenv("API_KEY")
        )

In [ ]:
def embed(question: str) -> list[float]:
    resp = requests.post(
    "http://172.17.11.82:8000/embed/dense",
    json={"texts": [question]})
    
    return resp.json()["embeddings"][0]

def retrieve_question(question: str, collection: str,  k: int = 3) -> list[str]:   

    embedding = embed(question)

    results = client.query_points(
        collection_name=collection,
        query=embedding,
        limit=k
    )

    return [i.payload["text"] for i in results.points]

def RAG_LLM(question: str, collection: str,  k: int = 3) -> str:
    
    context = "\n".join(retrieve_question(question, collection, k))
    prompt = f"""Answer using the context only.

                    Context:
                    {context}

                    Question:
                    {question}
                    """
    try:
        response = client_opeai.chat.completions.create(
                    model="gemma-3-pass",
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant."},
                        {"role": "user", "content": prompt}
                    ],
                    stream=False
                )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error communicating with model server: {e}")


In [ ]:
question = "tell me about environment"
collection = "assessment"

In [ ]:
retrieve_question(question, collection)

In [ ]:
RAG_LLM(question, collection)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_classic.vectorstores import Chroma
from langchain_core.documents import Document
import copy
import numpy as np

document_splitter = CharacterTextSplitter(separator = ".",
                                          chunk_size = 500,
                                          chunk_overlap = 50)

#pdf
pdf_file = PyPDFLoader(f"E:\AI Projects\AI Course\AI_Files\Introduction_to_Data_and_Data_Science.pdf")
pdf_load = pdf_file.load()
#avoid all new line characters \n to decrease API cost
pdf_cut = copy.deepcopy(pdf_load) #to avoid over-riding the original file
for i in pdf_cut:
    i.page_content = ' '.join(i.page_content.split())
pdf_cut_split = document_splitter.split_documents(pdf_cut)

In [ ]:
pdf_cut_split

In [ ]:
import requests
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from qdrant_client.models import VectorParams, Distance, PointStruct


url = "http://127.0.0.1:8000/upload-docx"

files = {"file": open(r"E:\AI Projects\doc_files\CE1_EN.docx", "rb")}
response = requests.post(url, files=files)
paragraph = response.json()["content"]

print(paragraph)

In [ ]:
text = "\n".join(paragraph)
print(text)

In [ ]:
document_splitter.split_text(text)

In [1]:
import requests

# Assuming LM Studio is running on the default port
LM_STUDIO_URL = "http://localhost:1234/v1/embeddings"

In [2]:
x = "nima" 
payload = {
        "input": x,
        "model": "text-embedding-nomic-embed-text-v1.5", 
    }
response = requests.post(LM_STUDIO_URL, json=payload)

In [3]:
response.json()

{'object': 'list',
 'data': [{'object': 'embedding',
   'embedding': [0.008190002292394638,
    0.021685415878891945,
    -0.1320003867149353,
    0.0016883016796782613,
    0.0241409819573164,
    -0.04113566875457764,
    0.05616968125104904,
    -0.03308526799082756,
    0.001805886160582304,
    0.051967818289995193,
    -0.037397656589746475,
    0.048975978046655655,
    0.009359094314277172,
    -0.0170820914208889,
    -0.03930307924747467,
    -0.09805095195770264,
    -0.01493499893695116,
    0.0032348670065402985,
    -0.0030848244205117226,
    0.07783559709787369,
    0.010283548384904861,
    -0.03360944241285324,
    -0.0052819629199802876,
    -0.015453401021659374,
    0.13680419325828552,
    -0.04438004642724991,
    0.06055830046534538,
    -0.012041011825203896,
    0.04539121687412262,
    0.05347876250743866,
    0.061517443507909775,
    -0.036361198872327805,
    0.03037457913160324,
    0.03876371681690216,
    -0.008592004887759686,
    -0.023950854316353798